# Week 7 - Dimensionality Reduction (2/2)
**Notebook 2 : Latent Semantic Analysis (SVD) and t-SNE**

**Unstructured Data Analysis (2026-2)** · Professor: Misuk Kim · Teaching Assistant: Hojin Son

Notebook 1 kept some of the original columns. This notebook builds **new** columns: a few hundred latent dimensions
computed from the whole term-document matrix with the **singular value decomposition** (chapter 3 of lecture 6-2 / 7-1),
and then two dimensions made for the eye with **t-SNE** (chapter 4).

| Lecture chapter | What we do in code |
|---|---|
| 3. SVD and its properties | decompose the lecture's 10 x 10 matrix with NumPy, check $U^\top U = I$, count the rank, truncate |
| 3. Latent Semantic Indexing | `TruncatedSVD` on the TF-IDF matrix of 2,000 reviews: what the dimensions mean, similar documents, similar words, a 2-D picture |
| 4. Stochastic Neighbor Embedding | the probability $p_{j\|i}$, the radius $\sigma_i$ that gives a chosen perplexity, the KL cost |
| 4. t-SNE | the crowding problem and the Student-t tails; `TSNE` on the LSA vectors; perplexity 5 / 30 / 50; a map of words |

> **How to use this notebook in Colab**: `File ▸ Save a copy in Drive`, then run the cells from top to bottom with
> `Shift + Enter`. Run the **Setup** cell first. This is a new Colab session: it does not see Notebook 1's variables.

## &nbsp;0. Setup
The same 2,000 movie reviews as Notebook 1, this time as a **TF-IDF** matrix (Week 4), because LSA works on weights, not on 0 / 1.

In [ ]:
import nltk
for r in ['movie_reviews', 'stopwords']:
    nltk.download(r, quiet=True)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from nltk.corpus import movie_reviews, stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics.pairwise import cosine_similarity, euclidean_distances
from sklearn.manifold import TSNE

%config InlineBackend.figure_format = 'retina'
%matplotlib inline
pd.set_option('display.width', 140)
np.set_printoptions(linewidth=140, suppress=True)

english_stops = set(stopwords.words('english'))
fileids = movie_reviews.fileids()
docs = [' '.join(movie_reviews.words(f)) for f in fileids]
y = np.array([1 if f.startswith('pos') else 0 for f in fileids])          # 1 = Pos, 0 = Neg

tfidf = TfidfVectorizer(min_df=5, token_pattern=r'[a-z]{2,}', stop_words=list(english_stops))
X = tfidf.fit_transform(docs)                                               # documents x terms
terms = tfidf.get_feature_names_out()
print('TF-IDF matrix:', X.shape, ' (documents x terms)')

## &nbsp;1. SVD by hand, on the lecture's 10 x 10 matrix
Slide *Singular Value Decomposition*: any real $m \times n$ matrix $A$ factorizes as

$$A = U \, \Sigma \, V^\top, \qquad U: m \times m, \quad \Sigma: m \times n, \quad V^\top: n \times n$$

We use the same artificial term-document matrix as Notebook 1 (10 terms x 10 documents, 6 positive, 4 negative),
with **terms as rows** like the slide.

In [ ]:
A = np.array([
    [1, 1, 1, 1, 1, 1, 0, 0, 0, 0],   # Term 1
    [0, 0, 0, 0, 0, 0, 1, 1, 1, 1],   # Term 2
    [1, 1, 1, 1, 1, 1, 1, 1, 1, 1],   # Term 3
    [1, 1, 1, 1, 1, 1, 1, 1, 0, 0],   # Term 4
    [0, 0, 0, 1, 1, 1, 1, 1, 1, 1],   # Term 5
    [1, 1, 1, 0, 0, 0, 0, 0, 0, 0],   # Term 6
    [0, 0, 0, 0, 0, 0, 1, 1, 0, 0],   # Term 7
    [1, 0, 1, 0, 1, 0, 1, 0, 1, 0],   # Term 8
    [1, 1, 1, 0, 0, 0, 1, 0, 0, 0],   # Term 9
    [1, 0, 0, 0, 0, 0, 0, 0, 1, 1],   # Term 10
], dtype=float)
toy_y = np.array([1, 1, 1, 1, 1, 1, 0, 0, 0, 0])

U, s, Vt = np.linalg.svd(A)              # full SVD; s holds the singular values, Vt is V transposed
Sigma = np.zeros(A.shape)
Sigma[:len(s), :len(s)] = np.diag(s)

print('U :', U.shape, '  Sigma :', Sigma.shape, '  V^T :', Vt.shape)
print('singular values :', s.round(3))

### &nbsp;1-1. Properties
Slide *Properties of SVD*: the singular vectors are orthogonal, $U^\top U = V^\top V = I$, and the number of
**positive** singular values is the rank of $A$.

In [ ]:
I = np.eye(10)
print('U^T U = I :', np.allclose(U.T @ U, I), '   V^T V = I :', np.allclose(Vt @ Vt.T, I))
print('A = U Sigma V^T :', np.allclose(U @ Sigma @ Vt, A))

positive = s > 1e-10                       # a singular value of 1e-16 is a rounding error, not a dimension
print('#positive singular values :', positive.sum(), '   np.linalg.matrix_rank(A) :', np.linalg.matrix_rank(A))

Three singular values are (numerically) zero, so the rank of $A$ is 7: three of the ten terms are linear combinations
of the others. Term 3 = Term 1 + Term 2, Term 4 = Term 1 + Term 7, and Term 1 = Term 5 + Term 6 - Term 2.

### &nbsp;1-2. Reduced SVDs
Slide *Reduces SVDs*. Three ways of dropping parts of the full decomposition that carry nothing or little:

| | $U$ | $\Sigma$ | $V^\top$ | drops |
|---|---|---|---|---|
| full | $m \times m$ | $m \times n$ | $n \times n$ | nothing |
| thin | $m \times n$ | $n \times n$ | $n \times n$ | the columns of $U$ that multiply the zero rows of $\Sigma$ |
| compact | $m \times r$ | $r \times r$ | $r \times n$ | the zero singular values too ($r$ = rank) |
| **truncated** | $m \times k$ | $k \times k$ | $k \times n$ | everything but the $k$ largest singular values: **an approximation** |

Only the truncated SVD loses information. How much, for each $k$?

In [ ]:
def truncated(U, s, Vt, k):
    return U[:, :k] @ np.diag(s[:k]) @ Vt[:k, :]       # A_k = U_k Sigma_k V_k^T


rows = []
for k in range(1, 11):
    A_k = truncated(U, s, Vt, k)
    rows.append({'k': k, 'sigma_k': round(s[k - 1], 3),
                 '||A - A_k||_F': round(np.linalg.norm(A - A_k), 3),
                 'variance kept': round((s[:k] ** 2).sum() / (s ** 2).sum(), 3)})
pd.DataFrame(rows).set_index('k').T

In [ ]:
A_2 = truncated(U, s, Vt, 2)
print('A_2 (k = 2), rounded:')
print(A_2.round(1))
print('\nthe original A:')
print(A.astype(int))

With $k = 2$ the reconstruction already shows the block structure of $A$ (values near 1 in the positive block of Term 1,
near 0 in its negative block), and $k = 7$ reproduces $A$ exactly. The squared singular values tell how much of the
matrix each dimension carries: the first two hold about 80 %.

### &nbsp;1-3. From $A_k$ to a $k \times n$ document matrix
Slide *Reduce the dimensions using SVD*: Step 1 build $A_k$, Step 2 multiply by $U_k^\top$ to get a $k \times n$ matrix
(one column per document, $k \ll m$ rows), Step 3 apply any data mining algorithm to it.

$$U_k^\top A = U_k^\top U_k \Sigma_k V_k^\top = \Sigma_k V_k^\top$$

So the documents in $k$ dimensions are the rows of $V_k$ scaled by the singular values. Let us look at the ten documents in 2-D.

In [ ]:
k = 2
D2 = (U[:, :k].T @ A).T                 # n x k : one row per document (same as (Sigma_k V_k^T).T)
print('U_k^T A  -> documents x k :', D2.shape)
print(np.allclose(D2, (np.diag(s[:k]) @ Vt[:k, :]).T))

plt.figure(figsize=(5, 4))
for c, color, name in [(1, 'tab:blue', 'Pos'), (0, 'tab:red', 'Neg')]:
    plt.scatter(D2[toy_y == c, 0], D2[toy_y == c, 1], c=color, label=name, s=60)
for i in range(10):
    plt.annotate(f'D{i + 1}', (D2[i, 0], D2[i, 1]), xytext=(4, 4), textcoords='offset points')
plt.xlabel('dimension 1'); plt.ylabel('dimension 2'); plt.legend(); plt.title('10 documents, 2 latent dimensions')
plt.show()

Nobody told the SVD which documents are positive, and yet the second dimension separates D1-D6 from D7-D10:
the label is the largest source of structure in this tiny matrix. On real data it rarely is, as we see next.

## &nbsp;2. Latent Semantic Analysis on 2,000 reviews
Slide *Latent Semantic Indexing*: apply the truncated SVD to the term-document matrix, so that a document is no longer
described by 13,000 word weights but by $k$ **latent** dimensions. *Latent* because the associations are not explicit,
*semantic* because meaning is represented as similarity to other documents and words.

scikit-learn's `TruncatedSVD` does Steps 1 and 2 in one call, works **directly on the sparse matrix**, and uses the
documents-as-rows convention: its input is $A^\top$, its `fit_transform` returns the $n \times k$ document matrix
(the transpose of the lecture's $U_k^\top A$), and `components_` holds the $k$ term directions (the lecture's $U_k^\top$).

In [ ]:
k = 100
svd = TruncatedSVD(n_components=k, random_state=7)
D = svd.fit_transform(X)                  # documents x k
T = svd.components_.T * svd.singular_values_   # terms x k : the words in the same latent space

print('documents x k :', D.shape, '   terms x k :', T.shape)
print('singular values (first 5):', svd.singular_values_[:5].round(2))
print(f'variance kept by {k} dimensions: {svd.explained_variance_ratio_.sum():.1%}')

A hundred numbers per review instead of 13,000. They keep less than a fifth of the variance (TF-IDF matrices are very
spread out), and we will see that this is enough for similarity.

### &nbsp;2-1. What does a latent dimension mean?
Slide *SVD in text mining* lists, for each SVD dimension of 41,717 NSF abstracts, the *top 10 positive and negative keywords*.
A dimension is a direction in word space, so the words with the largest positive and the largest negative weights name it.

In [ ]:
def keywords(dim, n=8):
    w = svd.components_[dim]
    pos = terms[np.argsort(-w)[:n]]
    neg = terms[np.argsort(w)[:n]]
    return ', '.join(pos), ', '.join(neg)


for dim in range(5):
    pos, neg = keywords(dim)
    print(f'dimension {dim + 1}')
    print('   +', pos)
    print('   -', neg)

Dimension 1 is the "average review": its positive side is the most common words, its negative side is noise (rare words
with weights near zero). From dimension 2 on, a dimension **contrasts two themes**: science-fiction and action
(`alien`, `effects`, `wars`, `planet`) against family drama and comedy (`life`, `love`, `family`, `father`); Jackie Chan
comedies against *Star Wars*; Hong Kong action against slasher horror (`scream`, `killer`, `williamson`); horror against
Disney animation (`tarzan`, `mulan`). None of them is *good vs bad*: the sentiment is spread across many small dimensions.
That is what the lecture calls latent structure, and what topic models (LDA, later in the course) make explicit.

### &nbsp;2-2. Similar documents : TF-IDF vs the latent space
Take one review, find its five nearest neighbors by cosine similarity in both spaces.

In [ ]:
def show_neighbors(sim, query, n=5):
    for rank, i in enumerate([i for i in np.argsort(-sim) if i != query][:n], 1):
        print(f'   {rank}. cos={sim[i]:.3f}  [{"pos" if y[i] else "neg"}]  {docs[i][:75]}')


QUERY = 1300
print('QUERY:', f'[{"pos" if y[QUERY] else "neg"}]', docs[QUERY][:100], '\n')

sim_tfidf = cosine_similarity(X[QUERY], X)[0]
print('TF-IDF (13,037 dimensions)')
show_neighbors(sim_tfidf, QUERY)

sim_lsa = cosine_similarity(D[QUERY:QUERY + 1], D)[0]
print('\nLSA (100 dimensions)')
show_neighbors(sim_lsa, QUERY)

TF-IDF finds the other review of *The Green Mile* (shared rare words), then a review of a **Tom Green** comedy: the word
`green`. The latent space finds the second *Green Mile* review too, then three other **Stephen King** adaptations
(*Dolores Claiborne*, ...): a theme, not a word. And its cosines are far higher: TF-IDF cosines between long documents
are small because most words do not overlap.

Is this true for every review, not only for one? We reuse Week 6's label-free check: for every review, do its 10 nearest
neighbors share its label? (0.5 = chance.)

In [ ]:
def same_label_share(M, n=10):
    sim = cosine_similarity(M)
    np.fill_diagonal(sim, -1)
    nn = np.argsort(-sim, axis=1)[:, :n]
    return (y[nn] == y[:, None]).mean()


rows = [('TF-IDF', X.shape[1], same_label_share(X))]
for kk in [10, 50, 100, 300]:
    Dk = TruncatedSVD(n_components=kk, random_state=7).fit_transform(X)
    rows.append((f'LSA k={kk}', kk, same_label_share(Dk)))
pd.DataFrame(rows, columns=['representation', 'dimensions', 'same label @10']).round(3)

Even **ten** latent dimensions do as well as 13,037 columns, fifty do best, and three hundred drift back toward TF-IDF
(the small dimensions bring the word-overlap noise back). This is the lecture's promise: *reduce the number of features*
while *improving the quality of text mining results*, at a fraction of the storage.

### &nbsp;2-3. Similar words
The same decomposition places the **words** in the latent space (`T`, terms x k). Two words are close when they occur in
the same kinds of documents, so LSA finds related words without any word embedding model.

In [ ]:
def similar_words(word, n=8):
    i = tfidf.vocabulary_[word]
    sim = cosine_similarity(T[i:i + 1], T)[0]
    return [terms[j] for j in np.argsort(-sim)[1:n + 1]]


for w in ['alien', 'vampire', 'hilarious', 'oscar']:
    print(f'{w:<10}', similar_words(w))

Compare with Week 6: Word2Vec learned `actor` ~ `actress` from the words *around* `actor`; LSA says `alien` ~ `ripley`,
`sigourney`, `weaver` because they appear in the **same reviews**. Document co-occurrence instead of window co-occurrence,
and the lecture's warning applies, *"the semantic distribution of each word is limited to the given corpus"*: in these
reviews `oscar` is not the award but Oscar Madison of *The Odd Couple II* (`matthau`, `felix`, `lemmon`, `grumpy`).

### &nbsp;2-4. A 2-D picture : the first two latent dimensions
Slide *Visualize the project in the reduced 2-D space*. The simplest picture is to keep dimensions 1 and 2.

In [ ]:
def scatter(Z, title, ax=None):
    ax = ax or plt.gca()
    for c, color, name in [(0, 'tab:red', 'neg'), (1, 'tab:blue', 'pos')]:
        ax.scatter(Z[y == c, 0], Z[y == c, 1], s=6, c=color, alpha=0.5, label=name)
    ax.set_title(title); ax.set_xticks([]); ax.set_yticks([]); ax.legend(markerscale=3)


plt.figure(figsize=(6, 5))
scatter(D[:, :2], 'LSA : dimensions 1 and 2')
plt.show()

Positive and negative reviews sit on top of each other. Dimensions 1 and 2 are the *largest* directions of the data,
not the most *informative* ones for our eyes: with 2,000 points and 100 dimensions, no linear picture shows the neighborhoods.
That is the job of t-SNE.

## &nbsp;3. Stochastic Neighbor Embedding
Slide *Stochastic Neighbor Embedding*: *"it is more important to get local distances right than non-local ones"*.
SNE turns every pairwise distance in the high-dimensional space into the probability that point $i$ would pick point $j$
as its neighbor,

$$p_{j|i} = \frac{\exp(-\lVert x_i - x_j \rVert^2 / 2\sigma_i^2)}{\sum_{k \neq i} \exp(-\lVert x_i - x_k \rVert^2 / 2\sigma_i^2)}$$

and asks the 2-D map to produce the same probabilities. Close points get almost all of the probability; far points get
practically none, so they hardly matter.

### &nbsp;3-1. Picking the radius $\sigma_i$ : perplexity
Slide *Picking the Radius of the Gaussian*: a big $\sigma_i$ spreads the probability over many neighbors (high entropy),
a small one concentrates it on a few. SNE fixes the **entropy** instead of the radius, through the *perplexity*
$Perp(P_i) = 2^{H(P_i)}$, a smooth "effective number of neighbors", and finds the $\sigma_i$ that produces it for every
point separately, by binary search. We do that for a few reviews in the 100-dimensional LSA space.

In [ ]:
dist2 = euclidean_distances(D, squared=True)                  # squared distances between all 2,000 reviews


def p_cond(i, sigma):
    # p_{j|i} for one point i and one radius sigma
    p = np.exp(-dist2[i] / (2 * sigma ** 2))
    p[i] = 0
    return p / p.sum()


def perplexity(p):
    nz = p[p > 0]
    H = -(nz * np.log2(nz)).sum()
    return 2 ** H


def find_sigma(i, target=30, tol=1e-3):
    lo, hi = 1e-4, 10.0
    for _ in range(100):                      # binary search on sigma
        mid = (lo + hi) / 2
        if perplexity(p_cond(i, mid)) > target:
            hi = mid
        else:
            lo = mid
        if hi - lo < tol:
            break
    return mid


for i in [0, 500, 1300, 1999]:
    sig = find_sigma(i, 30)
    p = p_cond(i, sig)
    top = np.sort(p)[::-1]
    print(f'review {i:>4}: sigma = {sig:.3f}   perplexity = {perplexity(p):5.1f}   '
          f'mass on the 30 nearest = {top[:30].sum():.2f}   on the 300 nearest = {top[:300].sum():.3f}')

Same perplexity, different radius for every review: a review in a dense region gets a small $\sigma_i$, an isolated one
a large $\sigma_i$, and both end up with about thirty effective neighbors. The 300 nearest reviews hold almost all of the
probability; the other 1,700 are nearly invisible to the cost function.

The map is then found by moving the 2-D points to minimize the **Kullback-Leibler divergence** between the two
distributions (slide *Cost Function*), by gradient descent:

$$C = \sum_i KL(P_i \| Q_i) = \sum_i \sum_j p_{j|i} \log \frac{p_{j|i}}{q_{j|i}}$$

KL is not symmetric: a large $p_{j|i}$ represented by a small $q_{j|i}$ (close points drawn far apart) costs a lot,
the opposite costs little. That is how "local distances first" is built into the cost.

### &nbsp;3-2. The crowding problem, and the t in t-SNE
Slide *Crowding problem*: in 2-D there is not enough room around a point for all of its moderately distant neighbors,
so they are squeezed together. t-SNE's remedy (slide *Resolution to the Crowding Problem*): in the low-dimensional map,
convert distances with a **Student-t distribution with one degree of freedom** instead of a Gaussian,

$$q_{ij} = \frac{(1 + \lVert y_i - y_j \rVert^2)^{-1}}{\sum_{k \neq l} (1 + \lVert y_k - y_l \rVert^2)^{-1}}$$

whose heavy tails let a moderately distant point sit much farther away at no cost.

In [ ]:
d = np.linspace(0, 6, 300)
gauss = np.exp(-d ** 2 / 2)
student = 1 / (1 + d ** 2)

plt.figure(figsize=(6, 3.5))
plt.plot(d, gauss, label='Gaussian  exp(-d^2 / 2)   (high-dim, p)')
plt.plot(d, student, label='Student-t, 1 d.f.  1 / (1 + d^2)   (map, q)')
plt.xlabel('distance in the map'); plt.ylabel('unnormalized similarity'); plt.legend(); plt.grid(alpha=0.3)
plt.title('heavier tails: a point at distance 3 is not "almost zero"')
plt.show()
print('at d = 3 :  Gaussian', f'{np.exp(-9 / 2):.4f}', '  Student-t', f'{1 / 10:.4f}')

### &nbsp;3-3. t-SNE on the LSA vectors
scikit-learn's `TSNE` takes the 100 LSA dimensions (the recommended input: t-SNE on 13,000 sparse columns is slow and noisy),
`perplexity=30`, and starts from the PCA / SVD picture (`init='pca'`). It takes a few seconds.

In [ ]:
tsne = TSNE(n_components=2, perplexity=30, init='pca', random_state=7)
Z = tsne.fit_transform(D)
print('map:', Z.shape, '  final KL divergence:', round(tsne.kl_divergence_, 3))

fig, axes = plt.subplots(1, 2, figsize=(11, 5))
scatter(D[:, :2], 'LSA : dimensions 1 and 2', axes[0])
scatter(Z, 't-SNE of the 100 LSA dimensions (perplexity 30)', axes[1])
plt.show()

Same reviews, same colors. Neither picture is red-vs-blue, and the t-SNE map is not a set of big clusters either: 2,000
reviews of about 2,000 different films have no five "topics". What t-SNE adds is the **small, tight clumps** around the
edge. Let us see what they are: the reviews whose 4 nearest neighbors in the map are the closest together.

In [ ]:
d_map = euclidean_distances(Z)
np.fill_diagonal(d_map, np.inf)
radius = np.sort(d_map, axis=1)[:, 3]                 # distance to the 4th nearest neighbor in the map

for i in np.argsort(radius)[:24:8]:                   # three of the tightest clumps
    print(f'--- clump around review {i}')
    for j in [i] + list(np.argsort(d_map[i])[:4]):
        print(f'   [{"pos" if y[j] else "neg"}]  {docs[j][:75]}')

Five reviews of *Shrek* (and *Antz*), five of *Spice World*, five of *Varsity Blues* (the films can differ a little
between scikit-learn versions): the clumps are **the same film, or the same kind of film**, reviewed by different people,
positive and negative together. t-SNE preserved the **local** structure of
the 100-dimensional space; the label was never part of it.

Now the same check with numbers: the share of same-label neighbors among the 10 nearest, in 100-D and in the 2-D map.

In [ ]:
def same_label_share_euclid(Z, n=10):
    d2 = euclidean_distances(Z, squared=True)
    np.fill_diagonal(d2, np.inf)
    nn = np.argsort(d2, axis=1)[:, :n]
    return (y[nn] == y[:, None]).mean()


print(f'same label @10  LSA 100-D : {same_label_share(D):.3f}')
print(f'same label @10  t-SNE 2-D : {same_label_share_euclid(Z):.3f}')
print(f'same label @10  LSA dims 1-2 only : {same_label_share_euclid(D[:, :2]):.3f}')

Two t-SNE dimensions keep most of the neighborhood information of a hundred (0.59 vs 0.62, chance is 0.50); the first two
SVD dimensions keep much less (0.55). That is what *"preserve local distances"* buys, and also its limit: the map is for
looking, not for training a model (a new review cannot be placed on it without recomputing everything).

### &nbsp;3-4. Perplexity 5 / 30 / 50
Slide: *"The performance of SNE is fairly robust to changes in the perplexity (5 - 50)"*. Three maps, three values.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, perp in zip(axes, [5, 30, 50]):
    t = TSNE(n_components=2, perplexity=perp, init='pca', random_state=7)
    Zp = t.fit_transform(D)
    scatter(Zp, f'perplexity {perp}   (KL {t.kl_divergence_:.2f})', ax)
plt.show()

Perplexity 5 breaks the map into many small islands (every point looks at five neighbors only), 50 draws a smoother
cloud; the same tight clumps are visible in all three. Do not read the KL values across perplexities as "better or worse":
they are costs of different problems.

### &nbsp;3-5. A map of words
The latent space also holds the words (`T`). A t-SNE of the 400 most frequent words, written out, is the picture you
usually see for word embeddings; the same code works on Week 6's `w2v.wv.vectors`.

In [ ]:
df_counts = np.asarray((X > 0).sum(axis=0)).ravel()
top = np.argsort(-df_counts)[:400]
Zw = TSNE(n_components=2, perplexity=20, init='pca', random_state=7).fit_transform(T[top])

plt.figure(figsize=(13, 10))
plt.scatter(Zw[:, 0], Zw[:, 1], s=4, c='lightgray')
for (x0, y0), w in zip(Zw, terms[top]):
    plt.text(x0, y0, w, fontsize=7)
plt.xticks([]); plt.yticks([]); plt.title('t-SNE of 400 words in the 100-dimensional LSA space')
plt.show()

Look for the islands: names of genres, words about acting, words about the plot, the negative words (`bad`, `boring`, `worst`)
close together. The axes mean nothing, the distances between islands mean little; only the neighborhoods are to be trusted.

## Summary

* **SVD** factors any matrix into $U \Sigma V^\top$; keeping the $k$ largest singular values gives the best rank-$k$
  approximation, and $U_k^\top A = \Sigma_k V_k^\top$ is the document matrix in $k$ dimensions.
* **LSA** = truncated SVD of the term-document matrix. 50 - 100 latent dimensions of 2,000 reviews keep the neighborhoods
  of 13,000 columns, give higher and more meaningful similarities, and place words and documents in one space.
  Its dimensions are themes, not the class.
* **SNE** matches neighbor probabilities; the perplexity fixes the effective number of neighbors and the radius adapts to
  every point; the cost is a KL divergence that punishes separating close points.
* **t-SNE** uses Student-t tails in the map to solve the crowding problem. It preserves local structure for the eye, is robust
  to the perplexity in 5 - 50, and is not a representation to train on.
* Together with Notebook 1: selection keeps columns (filter / wrapper), extraction builds them (LSA), and t-SNE draws them.